# 02 — Preprocessing: Model Perencana Makan

**Augmentasi wajib (dari paper IPB Jurnal 2024 + Frontiers Nutrition 2025):**
1. `category` — heuristic keyword 3x lebih kaya
2. `cuisine` — PADANG/JAVA/SUNDA/ASIAN/WESTERN/GENERAL_INDO
3. `estimated_price_idr` — heuristic per kategori × kalori
4. `is_halal`, `is_vegetarian`, `is_vegan`, `is_gluten_free`
5. Impute 0-kalori → 1
6. `slug` unique URL-safe

**Output:** `output/preprocessed/food_master.parquet` (1,346 row × 18 kolom — siap seed ke MySQL)

In [1]:
import pandas as pd
import numpy as np
import re
import os
import json

np.random.seed(42)
os.makedirs('output/preprocessed', exist_ok=True)

NUT_PATH = '../../dataset/Model_Perencana Makan_dan_Nutrisi/Indonesian Food & Drink Nutrition Dataset/nutrition.csv'
df = pd.read_csv(NUT_PATH)
print(f'Loaded: {df.shape}')

Loaded: (1346, 7)


In [2]:
# ── 1. Kategorisasi Heuristic (Enhanced) ──
KEYWORDS = {
    'PROTEIN':    ['ayam','sapi','ikan','telur','tahu','tempe','daging','udang','cumi',
                   'bebek','kambing','lele','bandeng','gurame','salmon','tuna','bakso',
                   'keju','sosis','kornet','sarden','ati','hati','jerohan','rendang','semur'],
    'VEGETABLE':  ['sayur','bayam','kangkung','brokoli','wortel','sawi','kol','daun',
                   'buncis','kubis','terong','labu','timun','selada','bawang','tomat',
                   'gado-gado','urap','pecel','asinan','karedok','lalap','sayuran'],
    'FRUIT':      ['buah','apel','mangga','pisang','jeruk','semangka','anggur','pepaya',
                   'salak','melon','nanas','alpukat','lemon','jambu','rambutan','duku',
                   'durian','manggis','kelengkeng','sirsak','markisa','strawberry','kiwi'],
    'BEVERAGE':   ['minum','teh','kopi','jus','susu','air ','minuman','sirup','soda',
                   'es teh','es jeruk','cappuccino','latte','smoothie','milkshake',
                   'wedang','jamu','bandrek','sekoteng','cendol'],
    'DESSERT':    ['kue','puding','es krim','dodol','klepon','bakpia','cake','martabak',
                   'brownie','coklat','permen','gulali','halua','tart','onde','getuk',
                   'wajik','lemper','lapis','bika ambon','pancake'],
    'SNACK':      ['keripik','biskuit','kerupuk','nugget','chiki','crackers',
                   'gorengan','bakwan','tahu isi'],
    'STAPLE':     ['nasi','mie','kentang','jagung','roti','pasta','ubi','singkong',
                   'bihun','ketupat','lontong','bubur','oatmeal','sereal','spaghetti'],
}

def categorize(name):
    n = str(name).lower()
    for cat in ['PROTEIN','VEGETABLE','FRUIT','BEVERAGE','DESSERT','SNACK','STAPLE']:
        if any(kw in n for kw in KEYWORDS[cat]):
            return cat
    return 'STAPLE'

df['category'] = df['name'].apply(categorize)
print('Kategori:')
print(df['category'].value_counts())

Kategori:
category
STAPLE       667
PROTEIN      321
VEGETABLE    157
FRUIT        100
DESSERT       49
BEVERAGE      29
SNACK         23
Name: count, dtype: int64


In [3]:
# ── 2. Cuisine Detection ──
CUISINE_KEYWORDS = {
    'INDONESIAN_PADANG': ['rendang','dendeng','sate padang','gulai','kalio','lemang',
                          'sambal lado','gajeboh','asam pedas','pangek'],
    'INDONESIAN_JAVA':   ['gudeg','soto','rawon','tumpeng','wedang','klepon','onde',
                          'getuk','tahu petis','rujak cingur','lontong balap'],
    'INDONESIAN_SUNDA':  ['karedok','lalap','nasi tutug','nasi liwet','batagor','siomay',
                          'pepes','sayur asem','ulukutek'],
    'ASIAN':             ['ramen','sushi','dimsum','bakpao','mie ayam','kwetiau','nasi goreng',
                          'capcay','fuyunghai'],
    'WESTERN':           ['burger','pizza','steak','spaghetti','sandwich','french fries',
                          'pasta','salad'],
}

def detect_cuisine(name):
    n = str(name).lower()
    for cuisine, kws in CUISINE_KEYWORDS.items():
        if any(kw in n for kw in kws):
            return cuisine
    return 'INDONESIAN_GENERAL'

df['cuisine'] = df['name'].apply(detect_cuisine)
print('Cuisine:')
print(df['cuisine'].value_counts())

Cuisine:
cuisine
INDONESIAN_GENERAL    1282
INDONESIAN_PADANG       27
INDONESIAN_JAVA         24
INDONESIAN_SUNDA         8
WESTERN                  3
ASIAN                    2
Name: count, dtype: int64


In [4]:
# ── 3. Heuristic Price ──
BASE_PRICE = {
    'STAPLE': 8000, 'PROTEIN': 18000, 'VEGETABLE': 6000, 'FRUIT': 7000,
    'BEVERAGE': 5000, 'DESSERT': 10000, 'SNACK': 5000
}

# Multiplier untuk cuisine premium
CUISINE_MULT = {
    'INDONESIAN_PADANG': 1.10,  # Rendang dll. lebih mahal
    'INDONESIAN_JAVA': 1.00,
    'INDONESIAN_SUNDA': 0.95,
    'ASIAN': 1.15,
    'WESTERN': 1.30,
    'INDONESIAN_GENERAL': 1.00,
}

def estimate_price(row):
    base = BASE_PRICE.get(row['category'], 7000)
    cal = max(row['calories'], 10)
    cuisine_mult = CUISINE_MULT.get(row['cuisine'], 1.0)
    price = base * (1 + cal / 500) * cuisine_mult
    price = price * np.random.uniform(0.85, 1.15)
    return round(price / 500) * 500

df['estimated_price_idr'] = df.apply(estimate_price, axis=1)
print('Sample harga:')
print(df[['name','category','cuisine','calories','estimated_price_idr']].head(10))

Sample harga:
                 name category             cuisine  calories  \
0                Abon   STAPLE  INDONESIAN_GENERAL     280.0   
1        Abon haruwan   STAPLE  INDONESIAN_GENERAL     513.0   
2           Agar-agar   STAPLE  INDONESIAN_GENERAL       0.0   
3  Akar tonjong segar   STAPLE  INDONESIAN_GENERAL      45.0   
4       Aletoge segar   STAPLE  INDONESIAN_GENERAL      37.0   
5       Alpukat segar    FRUIT  INDONESIAN_GENERAL      85.0   
6  Ampas kacang hijau   STAPLE  INDONESIAN_GENERAL      96.0   
7          Ampas Tahu  PROTEIN  INDONESIAN_GENERAL     414.0   
8    Ampas tahu kukus  PROTEIN  INDONESIAN_GENERAL      75.0   
9   Ampas tahu mentah  PROTEIN  INDONESIAN_GENERAL      67.0   

   estimated_price_idr  
0                12000  
1                18500  
2                 8500  
3                 9000  
4                 7500  
5                 7500  
6                 8500  
7                36500  
8                21500  
9                21500  


In [5]:
# ── 4. Dietary Flags (Enhanced multi-token matching) ──
NON_HALAL = ['babi','pork','bacon','ham','lard','wine','bir','alkohol','brandy','rum',
             'whisky','sake','vodka']
NON_VEG   = ['daging','ayam','sapi','ikan','udang','cumi','bebek','kambing',
             'lele','bandeng','gurame','salmon','tuna','bakso','kornet','sarden',
             'kornet','sosis','ati','hati','jerohan','semur','rendang']
NON_VEGAN = NON_VEG + ['susu','keju','mentega','butter','cream','telur','yoghurt',
                      'krim','madu','keju']
GLUTEN    = ['roti','pasta','mie','gandum','tepung terigu','spaghetti','noodle']

def flag(name, keywords):
    n = str(name).lower()
    return not any(kw in n for kw in keywords)

df['is_halal']       = df['name'].apply(lambda n: flag(n, NON_HALAL))
df['is_vegetarian']  = df['name'].apply(lambda n: flag(n, NON_VEG))
df['is_vegan']       = df['name'].apply(lambda n: flag(n, NON_VEGAN))
df['is_gluten_free'] = df['name'].apply(lambda n: flag(n, GLUTEN))

for col in ['is_halal','is_vegetarian','is_vegan','is_gluten_free']:
    print(f'  {col}: {df[col].sum()} / {len(df)} ({df[col].mean()*100:.1f}%)')

  is_halal: 1327 / 1346 (98.6%)
  is_vegetarian: 1068 / 1346 (79.3%)
  is_vegan: 1033 / 1346 (76.7%)
  is_gluten_free: 1328 / 1346 (98.7%)


In [6]:
# ── 5. Impute 0-Kalori & Missing ──
print(f'0-kalori sebelum: {len(df[df["calories"]==0])}')
df['calories'] = df['calories'].replace(0, 1)
df['proteins'] = df['proteins'].fillna(0)
df['fat'] = df['fat'].fillna(0)
df['carbohydrate'] = df['carbohydrate'].fillna(0)
print(f'0-kalori sesudah: {len(df[df["calories"]==0])}')

0-kalori sebelum: 1
0-kalori sesudah: 0


In [7]:
# ── 6. Slug Generation ──
def make_slug(name):
    slug = str(name).lower()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    slug = re.sub(r'\s+', '-', slug.strip())
    return slug

df['slug'] = df['name'].apply(make_slug)
# Handle duplicates
slugs_seen = {}
new_slugs = []
for slug in df['slug']:
    if slug in slugs_seen:
        slugs_seen[slug] += 1
        new_slugs.append(f'{slug}-{slugs_seen[slug]}')
    else:
        slugs_seen[slug] = 0
        new_slugs.append(slug)
df['slug'] = new_slugs
print(f'Unique slugs: {df["slug"].nunique()} / {len(df)}')

Unique slugs: 1346 / 1346


In [8]:
# ── 7. Final Schema (match DB schema) ──
df_master = df[[
    'id', 'slug', 'name', 'category', 'cuisine',
    'calories', 'proteins', 'fat', 'carbohydrate',
    'estimated_price_idr',
    'is_halal', 'is_vegetarian', 'is_vegan', 'is_gluten_free',
    'image',
]].copy()

df_master = df_master.rename(columns={
    'proteins':     'protein_g',
    'fat':          'fat_g',
    'carbohydrate': 'carbs_g',
    'calories':     'calories_per_portion',
    'image':        'image_url',
})

df_master['base_portion'] = '1 porsi'
df_master['base_portion_gram'] = 100
df_master['fiber_g'] = 0.0
df_master['is_active'] = True

# Save
df_master.to_parquet('output/preprocessed/food_master.parquet', index=False)
df_master.to_csv('output/preprocessed/food_master.csv', index=False)

# Metadata
metadata = {
    'n_foods': len(df_master),
    'columns': list(df_master.columns),
    'category_counts': df_master['category'].value_counts().to_dict(),
    'cuisine_counts': df_master['cuisine'].value_counts().to_dict(),
    'halal_count': int(df_master['is_halal'].sum()),
    'vegetarian_count': int(df_master['is_vegetarian'].sum()),
    'augmentation_done': True,
}
with open('output/preprocessed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ food_master.parquet: {len(df_master)} baris × {len(df_master.columns)} kolom')
df_master.head(5)

✅ food_master.parquet: 1346 baris × 19 kolom


,id,slug,name,category,cuisine,calories_per_portion,protein_g,fat_g,carbs_g,estimated_price_idr,is_halal,is_vegetarian,is_vegan,is_gluten_free,image_url,base_portion,base_portion_gram,fiber_g,is_active
0,1,abon,Abon,STAPLE,INDONESIAN_GENERAL,280.0,9.2,28.4,0.0,12000,True,True,True,True,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...,1 porsi,100,0.0,True
1,2,abon-haruwan,Abon haruwan,STAPLE,INDONESIAN_GENERAL,513.0,23.7,37.0,21.3,18500,True,True,True,True,https://img-global.cpcdn.com/recipes/cbf330fbd...,1 porsi,100,0.0,True
2,3,agar-agar,Agar-agar,STAPLE,INDONESIAN_GENERAL,1.0,0.0,0.2,0.0,8500,True,True,True,True,https://res.cloudinary.com/dk0z4ums3/image/upl...,1 porsi,100,0.0,True
3,4,akar-tonjong-segar,Akar tonjong segar,STAPLE,INDONESIAN_GENERAL,45.0,1.1,0.4,10.8,9000,True,True,True,True,https://images.tokopedia.net/img/cache/200-squ...,1 porsi,100,0.0,True
4,5,aletoge-segar,Aletoge segar,STAPLE,INDONESIAN_GENERAL,37.0,4.4,0.5,3.8,7500,True,True,True,True,https://nilaigizi.com/assets/images/produk/pro...,1 porsi,100,0.0,True
